In [ ]:
import pandas as pd
import re
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import random
import joblib

[nltk_data] Downloading package wordnet to C:\Users\Kalpana
[nltk_data]     Kumari\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\Kalpana
[nltk_data]     Kumari\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt to C:\Users\Kalpana
[nltk_data]     Kumari\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\Kalpana Kumari\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


In [2]:
df=pd.read_csv('Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv')

In [3]:
def placeholder(text):
    return re.sub(r'\{\{(.*?)\}\}', lambda x: f'placeholder_{x.group(1).replace(" ", "_")}', text)

def cleaning(text):
    text=re.sub(r'\{\{.*?\}\}|[*?]', "", text)
    text=re.sub(r'[^\w\s]', "", text)
    return text.lower()

lm=WordNetLemmatizer()

def tag_helper(txt):
    if txt.startswith('J'):
        return wordnet.ADJ
    elif txt.startswith('V'):
        return wordnet.VERB
    elif txt.startswith('N'):
        return wordnet.NOUN
    elif txt.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN
    
def lemma_word(text):
    lm_setnz=[]
    tokens=nltk.word_tokenize(text)
    tagged=nltk.pos_tag(tokens)
    for word, tag in tagged:
        wn_tag=tag_helper(tag)
        cleaned_word=lm.lemmatize(word, pos=wn_tag)
        lm_setnz.append(cleaned_word)
    return ' '.join(lm_setnz)

tf=TfidfVectorizer(ngram_range=(1,2))
kmeans=KMeans(n_clusters=2, random_state=42, n_init='auto')
le=LabelEncoder()
tfidf=TfidfVectorizer(ngram_range=(1,2))
lr=LogisticRegression()

In [4]:
df['instruction']=df['instruction'].apply(cleaning).apply(lemma_word)

# df['instruction']=df['instruction'].apply(lemma_word)

In [5]:
df.isnull().sum()

flags          0
instruction    0
category       0
intent         0
response       0
dtype: int64

In [6]:
df_r=df[['intent', 'response']].copy()
df_r['cleaned_response']=df_r['response'].apply(placeholder).apply(cleaning)

In [7]:
df_i=df[['intent', 'instruction']].copy()
df_i

,intent,instruction
0,cancel_order,question about cancel order
1,cancel_order,i have a question about cancel oorder
2,cancel_order,i need help cancelling puchase
3,cancel_order,i need to cancel purchase
4,cancel_order,i can not afford this order cancel purchase
...,...,...
26867,track_refund,i be wait for a rebate of dollar
26868,track_refund,how to see if there be anything wrong with my ...
26869,track_refund,im wait for a reimbjrsement of
26870,track_refund,i dont know what to do to see my reimbursement...


In [8]:
golden_responses={}
for i in df_r['intent'].value_counts().index:
    data=df_r[df_r['intent']==i].copy()
    vec_cresponse=tf.fit_transform(data['cleaned_response'])

    data['kmeans_response']=kmeans.fit_predict(vec_cresponse)

    data['length']=data['response'].str.len()

    avg_len=data.groupby('kmeans_response')['length'].mean()

    cluster=avg_len.idxmax()

    golden_subset=data[data['kmeans_response']==cluster]

    response_pool=golden_subset['response'].unique()[:5].tolist()

    golden_responses[i]=response_pool


In [9]:
df_i

,intent,instruction
0,cancel_order,question about cancel order
1,cancel_order,i have a question about cancel oorder
2,cancel_order,i need help cancelling puchase
3,cancel_order,i need to cancel purchase
4,cancel_order,i can not afford this order cancel purchase
...,...,...
26867,track_refund,i be wait for a rebate of dollar
26868,track_refund,how to see if there be anything wrong with my ...
26869,track_refund,im wait for a reimbjrsement of
26870,track_refund,i dont know what to do to see my reimbursement...


In [10]:
X=df_i['instruction']
Y=df_i['intent']
print(X.head())
print(Y.head())

0                    question about cancel order
1          i have a question about cancel oorder
2                 i need help cancelling puchase
3                      i need to cancel purchase
4    i can not afford this order cancel purchase
Name: instruction, dtype: str
0    cancel_order
1    cancel_order
2    cancel_order
3    cancel_order
4    cancel_order
Name: intent, dtype: str


In [11]:
X_train,X_test,Y_train,Y_test=train_test_split(X,Y, test_size=0.2, random_state=42)

In [12]:
X_train=tfidf.fit_transform(X_train)
X_test=tfidf.transform(X_test)
Y_train=le.fit_transform(Y_train)
Y_test=le.transform(Y_test)

In [13]:
lr.fit(X_train, Y_train)
Y_predict=lr.predict(X_test)
classification=classification_report(Y_test,Y_predict)


In [14]:
def chat_with_bot(user_input):

    cleaned_text=cleaning(user_input)
    cleaned_text=lemma_word(cleaned_text)

    transformed_input=tfidf.transform([cleaned_text])

    probabilities = lr.predict_proba(transformed_input)
    
    max_confidence = probabilities.max()

    if max_confidence < 0.25:
        print("I'm sorry, I don't quite understand. Could you rephrase your question, or would you like to speak to a human agent?")
        return
    
    vec_intent=lr.predict(transformed_input)
    f_intent=le.inverse_transform(vec_intent)
    intent_1=f_intent[0]
    
    if intent_1 in golden_responses:
        return random.choice(golden_responses[intent_1])
    
    return "System Error: Intent recognized but no response mapped."
        

In [15]:
joblib.dump(lr, 'chatbot_model.pkl')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')
joblib.dump(le, 'label_encoder.pkl')
joblib.dump(golden_responses, 'golden_responses.pkl')

['golden_responses.pkl']